# Hard agentic-modeling challenges

_Investigation `agentic-challenges` — coder reproduction notebook._

**Question.** Which modeling tasks genuinely require an LLM agent — where a deterministic policy has no move for the real fix — and can the agent build a passing process-bigraph model for them?

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/Users/eranagmon/code/viva-casebook--complete-findings').is_dir():
    REPO = Path('/Users/eranagmon/code/viva-casebook--complete-findings')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_casebook.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Bounded goal-directed cell (`bounded-cell`)

**Question.** Build a bounded, goal-directed cell: on a finite nutrient pool and a rising temperature, produce a cell whose BIOMASS grows on the nutrient it CONSUMES (mass-conserving, Monod-saturating GROWTH), and whose VIABILITY holds while below its temperature tolerance then collapses once temperature exceeds it.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `final-model` | `viva_casebook.composites.bounded-cell` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_casebook.composites.bounded-cell`** — `spec_viva_casebook_composites_bounded_cell` (a plain, editable dict)


_composite spec file for `viva_casebook.composites.bounded-cell` not found under `viva_casebook/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: bounded-cell ===
STUDY = 'bounded-cell'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| growth | observable=growth | op == value within_tol |
| nutrient-depletion | observable=nutrient-depletion | op == value within_tol |
| conservation | observable=conservation | op == value within_tol |
| viability-cliff | observable=viability-cliff | op == value within_tol |
| viability-in-band | observable=viability-in-band | op == value within_tol |
| saturation | observable=saturation | op == value within_tol |


## Study: Diauxic growth (glucose→lactose) (`diauxie`)

**Question.** Build a cell that grows on a mixture of glucose and lactose with the DIAUXIC phenotype: it consumes glucose first, and only once glucose is exhausted does it switch to lactose — a sequential shift, not simultaneous co-consumption.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `final-model` | `viva_casebook.composites.diauxie` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_casebook.composites.diauxie`** — `spec_viva_casebook_composites_diauxie` (a plain, editable dict)


_composite spec file for `viva_casebook.composites.diauxie` not found under `viva_casebook/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: diauxie ===
STUDY = 'diauxie'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| glucose-consumed | observable=glucose-consumed | op == value within_tol |
| lactose-consumed | observable=lactose-consumed | op == value within_tol |
| growth | observable=growth | op == value within_tol |
| diauxic-order | observable=diauxic-order | op == value within_tol |


## Study: Multicellular differentiation phenotype (`multicellular`)

**Question.** Produce a multicellular tissue with a spatial differentiation gradient: cells near a signalling niche stay stem, distal cells differentiate.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `final-model` | `viva_casebook.composites.multicellular` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_casebook.composites.multicellular`** — `spec_viva_casebook_composites_multicellular` (a plain, editable dict)


_composite spec file for `viva_casebook.composites.multicellular` not found under `viva_casebook/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: multicellular ===
STUDY = 'multicellular'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| simulator-fit | observable=simulator-fit | op == value within_tol |
| multicellular | observable=multicellular | op == value within_tol |
| differentiation-gradient | observable=differentiation-gradient | op == value within_tol |


## Study: SBML model → COPASI → quality (`sbml`)

**Question.** Build an SBML kinetic model of a linear metabolic pathway A→B→C that COPASI can load and simulate to a valid steady state, with total mass conserved and the terminal product C accumulating as the dominant species.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `final-model` | `viva_casebook.composites.sbml` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_casebook.composites.sbml`** — `spec_viva_casebook_composites_sbml` (a plain, editable dict)


_composite spec file for `viva_casebook.composites.sbml` not found under `viva_casebook/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: sbml ===
STUDY = 'sbml'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| sbml-valid | observable=sbml-valid | op == value within_tol |
| copasi-loads | observable=copasi-loads | op == value within_tol |
| steady-state | observable=steady-state | op == value within_tol |
| mass-conserved | observable=mass-conserved | op == value within_tol |
| terminal-product | observable=terminal-product | op == value within_tol |


## Study: Multiscale coupling via a translator (`multiscale`)

**Question.** Couple a cell-scale metabolic model (which produces a metabolite FLUX) to a tissue-scale spatial DIFFUSION FIELD of that metabolite, so that the cell's secretion sources the field and a gradient forms around the cell — with mass conserved across the scale interface.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `final-model` | `viva_casebook.composites.multiscale` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_casebook.composites.multiscale`** — `spec_viva_casebook_composites_multiscale` (a plain, editable dict)


_composite spec file for `viva_casebook.composites.multiscale` not found under `viva_casebook/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: multiscale ===
STUDY = 'multiscale'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| cell-active | observable=cell-active | op == value within_tol |
| field-active | observable=field-active | op == value within_tol |
| coupled-gradient | observable=coupled-gradient | op == value within_tol |
| flux-conserved | observable=flux-conserved | op == value within_tol |


## Study: Ambiguous diagnosis (why is biomass low?) (`diagnosis`)

**Question.** A cell's biomass is far below target. Diagnose the cause from the observables and add the mechanism that fixes it, so the cell both grows (biomass >= 3.0) and stays alive (viability >= 0.5).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `final-model` | `viva_casebook.composites.diagnosis` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_casebook.composites.diagnosis`** — `spec_viva_casebook_composites_diagnosis` (a plain, editable dict)


_composite spec file for `viva_casebook.composites.diagnosis` not found under `viva_casebook/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: diagnosis ===
STUDY = 'diagnosis'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| growth | observable=growth | op == value within_tol |
| survives | observable=survives | op == value within_tol |


## Study: Bistable genetic switch (`bistable`)

**Question.** Build a bistable genetic switch: two mutually repressing genes A and B that settle into two DISTINCT stable states and latch to whichever gene starts ahead (a decisive toggle, not a graded midpoint).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `final-model` | `viva_casebook.composites.bistable` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_casebook.composites.bistable`** — `spec_viva_casebook_composites_bistable` (a plain, editable dict)


_composite spec file for `viva_casebook.composites.bistable` not found under `viva_casebook/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: bistable ===
STUDY = 'bistable'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| bistable | observable=bistable | op == value within_tol |
| decisive | observable=decisive | op == value within_tol |
